# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, Taylor_Green_flow, gravity_field, constant_scalar
from softmobility.classes.flowbodysolver import _rotation_matrix_from_Rodrigues_impl
from softmobility.classes.flowbodysolver import compute_Bortz_operator, rescale_orientation

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

In [2]:
BLUEACCENT="#3274B5" 
REDACCENT="#DA3B26"
GREY="#929292"

## Simulation of default parameters

### YAML file

In [3]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  spring_k: .5           
  radius: .33
  _spr_: 50
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [phi, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [-_spr_ * spring_k * phi, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [-phi/_rad_/radius, 0, 0]
    force: [gravity0, gravity1 + active_force * sin(phi/_rad_/radius), gravity2 + active_force * cos(phi/_rad_/radius)]
    torque: [_spr_ * spring_k * phi, 0, 0]
"""

In [4]:
yaml_data_rigid = """
# Variable Prefixes (Optional)
design_names:      
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  radius: .5
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [0, 0, 0]
    force: [-gravity0, -gravity1, -gravity2]
    torque: [0, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [0, 0, 0]
    force: [gravity0, gravity1, gravity2 + active_force]
    torque: [0, 0, 0]
"""

### Soft Body

In [5]:
mybody= SoftBody(yaml_data, verbose=True)
myrigidbody = SoftBody(yaml_data_rigid, verbose=False)

  Found variables: phi, radius, spring_k, gravity0, gravity1, gravity2, active_force
  3D field inputs:  ['gravity']
  Scalar inputs:    ['active_force']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '1']
      Orientation: ['phi', '0', '0']
      Coupling matrix C_H:
[['-1', '0', '0', '0'], ['0', '-1', '0', '0'], ['0', '0', '-1', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['-50*spring_k'], ['0'], ['0']]
    Sphere 1
      Radius: radius
      Position: ['0', '0', '-radius']
      Orientation: ['-phi/radius', '0', '0']
      Coupling matrix C_H:
[['1', '0', '0', '0'], ['0', '1', '0', 'sin(phi/radius)'], ['0', '0', '1', 'cos(phi/radius)'], ['0', '0', '0', '0'], ['0', '0', '0', '0'], ['0', '0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['50*spring_k'], ['0'], ['0']]


### Parameters of flow and gravity field

In [6]:
# Calculating force to have a swimming velocity of 1
tensors = myrigidbody.compute_fast_mobility(dofs=jnp.zeros(myrigidbody.Ndof))
FORCE_INTENSITY=1/tensors.M_H[2,-1]

In [7]:
# Create a TGV flow with with max vorticy = 1
FLOW = Taylor_Green_flow(omega=2)
# Create a gravity field 
GRAVITY = gravity_field(g=50.0)
# Create an active force scalar
FORCE = constant_scalar(value=FORCE_INTENSITY)

In [8]:
# parameters of time integration
FINAL_TIME = 4 * jnp.pi  
DT = 0.1
N_TRAJECTORIES = 7

Y_INIT_POSITIONS = np.linspace(np.pi/N_TRAJECTORIES/2, 2*np.pi, N_TRAJECTORIES, endpoint=False)
N_DT = int(FINAL_TIME / DT)


### Functions

In [9]:
rotation_matrix_from_Rodrigues = jax.jit(
    lambda rvec, Ndof: _rotation_matrix_from_Rodrigues_impl(rvec, Ndof), static_argnums=(1,)
)

In [10]:
def sixc_velocity(design, orientation, dofs, field_lab, scalar, u_lab, omega_lab, E_lab):
    rot_matrix, sixc_rot_matrix = rotation_matrix_from_Rodrigues(orientation, Ndof=mybody.Ndof)
    
    E_body = rot_matrix.T @ E_lab @ rot_matrix
    E_inf = jnp.array([E_body[0, 0], E_body[0, 1], E_body[0, 2], E_body[1, 1], E_body[1, 2]])

    field_body = rot_matrix.T @ field_lab
    input_vec = jnp.concatenate([field_body, jnp.array([scalar])])
    
    tensors = mybody.compute_mobility_problem(dofs, design)
    sixc_velocity = tensors.M_H @ input_vec + tensors.M_K @ dofs + tensors.C_E @ E_inf

    p_lab = jnp.block([u_lab, omega_lab, jnp.zeros(mybody.Ndof)])
    return sixc_rot_matrix @ sixc_velocity + p_lab

In [11]:
def _rollout_one(design, x_init):
    """Simulate one trajectory and return mean Z velocity."""
    dofs0        = jnp.ones(mybody.Ndof) * 1e-6   # jnp.zeros causes a singularity
    tensors     = mybody.compute_fast_mobility(dofs=dofs0, design=design)
    force_norm  = 1.0 / tensors.M_H[2,3] / FORCE_INTENSITY
    
    def step(carry, t):
        position, orientation, dofs = carry
        time = t * DT

        field_lab = GRAVITY.vector(position, time)
        scalar    = force_norm  * FORCE.value(position, time)
        u_lab     = FLOW.velocity(position, time)
        omega_lab, E_lab = FLOW.omega_rate_of_strain(position, time)
        bortz     = compute_Bortz_operator(orientation)

        # RK2
        p1          = sixc_velocity(design, orientation, dofs, field_lab, scalar, u_lab, omega_lab, E_lab)
        k1_pos, k1_ori, k1_dof = p1[:3], p1[3:6], p1[6:]

        pos_mid = position    + DT * k1_pos / 2
        ori_mid = orientation + DT * bortz @ k1_ori / 2
        dof_mid = dofs        + DT * k1_dof / 2

        field_lab2  = GRAVITY.vector(pos_mid, time + DT/2)
        scalar2    = force_norm  * FORCE.value(pos_mid, time + DT/2)
        u_lab2     = FLOW.velocity(pos_mid, time + DT/2)
        omega_lab2, E_lab2 = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        p2          = sixc_velocity(design, ori_mid, dof_mid, field_lab2, scalar2, u_lab2, omega_lab2, E_lab2)

        k2_pos, k2_ori, k2_dof = p2[:3], p2[3:6], p2[6:]

        position    = position    + DT * (k1_pos + k2_pos) / 2
        orientation = orientation + DT * bortz @ (k1_ori + k2_ori) / 2
        orientation = rescale_orientation(orientation)
        dofs        = dofs        + DT * (k1_dof + k2_dof) / 2

        return (position, orientation, dofs), None

    position    = jnp.array([np.pi/2, x_init, 0.0])
    orientation = jnp.zeros(3)
    dofs = dofs0.copy()

    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME   # mean Z velocity


In [12]:
def mean_velocity_objective(design):
    """Negative mean Z velocity across all initial positions (negative = minimize to maximize)."""
    velocities = jnp.stack([_rollout_one(design, x_init) for x_init in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)


grad_fn = jax.jit(jax.value_and_grad(mean_velocity_objective))

In [13]:
def gradient_descent(
    grad_fn,
    initial_design,
    lr           = 0.05,
    n_steps      = 1000,
    print_every  = 100,
    clip_min     = 0.0,
    clip_max     = 1.0,
    grad_clip    = 1.0,    # max gradient norm — set None to disable
):
    design = initial_design
    design_keys = mybody.design_variables
    loss, grad = None, None
    best_design, best_velocity = initial_design, -jnp.inf

    for step in range(n_steps):
        loss, grad = grad_fn(design)

        # --- NaN guard: stop and revert to best if things blow up ---
        if jnp.isnan(loss) or jnp.any(jnp.isnan(grad)):
            print(f"  !! NaN detected at step {step}, stopping. Best was step with velocity={float(best_velocity):.5f}")
            return best_design, float(best_velocity)

        # --- Gradient clipping ---
        if grad_clip is not None:
            grad_norm = jnp.linalg.norm(grad)
            grad      = jnp.where(grad_norm > grad_clip, grad * grad_clip / grad_norm, grad)

        design = design - lr * grad
        design = jnp.clip(design, clip_min, clip_max)

        velocity = float(-loss)
        if velocity > best_velocity:
            best_velocity = velocity
            best_design   = design

        if step % print_every == 0 or step == n_steps - 1:
            if len(design_keys)==len(design):
                params_str = "  ".join(f"{k}={float(v):.4f}" for k, v in zip(design_keys, design))
            else:
                params_str = "  ".join(f"d{i}={float(v):.4f}" for i, v in enumerate(design))
            print(f"step {step:4d}  velocity={velocity:.5f}  "
                f"|grad|={float(jnp.linalg.norm(grad)):.5f}  {params_str}")

    print(f"\nOptimal design:")
    for k, v in zip(design_keys, best_design):
        print(f"  {k}: {float(v):.4f}  (init: {float(initial_design[list(design_keys).index(k)]):.4f})")
    print(f"Mean velocity: {float(best_velocity):.5f}")

    return best_design, float(best_velocity)

### Optimization of design

In [14]:
initial_guess = 0.5 * jnp.ones(mybody.Ndesign)

optimal_design, optimal_velocity = gradient_descent(
    grad_fn,
    initial_guess,
    lr          = 0.01,
    n_steps     = 1000,
    print_every = 100,
    clip_min=0.01,
    clip_max=1.0,
)

step    0  velocity=0.95346  |grad|=1.00000  radius=0.4932  spring_k=0.4926
step  100  velocity=1.22952  |grad|=0.08640  radius=0.2989  spring_k=0.2864
step  200  velocity=1.23196  |grad|=0.02614  radius=0.2607  spring_k=0.3121
step  300  velocity=1.23235  |grad|=0.01508  radius=0.2448  spring_k=0.3232
step  400  velocity=1.23253  |grad|=0.01165  radius=0.2341  spring_k=0.3306
step  500  velocity=1.23265  |grad|=0.01090  radius=0.2250  spring_k=0.3370
step  600  velocity=1.23278  |grad|=0.01195  radius=0.2157  spring_k=0.3435
step  700  velocity=1.23295  |grad|=0.01523  radius=0.2048  spring_k=0.3512
step  800  velocity=1.23330  |grad|=0.02284  radius=0.1897  spring_k=0.3620
step  900  velocity=1.23425  |grad|=0.04015  radius=0.1655  spring_k=0.3803
step  999  velocity=1.23585  |grad|=0.01525  radius=0.1427  spring_k=0.4052

Optimal design:
  radius: 0.1427  (init: 0.5000)
  spring_k: 0.4052  (init: 0.5000)
Mean velocity: 1.23585


### Surfer

In [15]:
tensors = mybody.compute_fast_mobility(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)

TAU_ALIGN = 0.5/tensors.M_H[3,1]/50
print(TAU_ALIGN)

0.06342698


In [16]:
from jax.scipy.linalg import expm

E_Z = jnp.array([0.0, 0.0, 1.0])   # reference "upward" direction

def _rollout_surfer(tau, x_init):
    """Surfer agent: swims at speed 1, aligns with z @ exp(tau * grad_u)."""

    position = jnp.array([np.pi/2, x_init, 0.0])
    hatp     = E_Z.copy()   # initial swimming direction: upward

    def step(carry, t):
        position, hatp = carry
        time = t * DT

        u_lab              = FLOW.velocity(position, time)
        omega_lab, _       = FLOW.omega_rate_of_strain(position, time)
        grad_u             = FLOW.gradient(position, time)   # 3×3 matrix

        # --- Preferred direction ---
        n    = E_Z @ expm(tau * grad_u)         # deform current heading
        hatn = n / jnp.linalg.norm(n)           # normalize

        # --- hatp ODE (on the unit sphere) ---
        #   d hatp/dt = omega × hatp + (hatn - (hatn·hatp) hatp) / (2 talign)
        vorticity_term  = jnp.cross(omega_lab, hatp)
        alignment_term  = (hatn - jnp.dot(hatn, hatp) * hatp) / (2.0 * TAU_ALIGN)
        dhatp           = vorticity_term + alignment_term

        # --- Position ODE: advected by flow + swimming at speed 1 ---
        dpos = u_lab + hatp

        # --- RK2 ---
        pos_mid  = position + DT * dpos  / 2
        hatp_mid = hatp     + DT * dhatp / 2
        hatp_mid = hatp_mid / jnp.linalg.norm(hatp_mid)   # keep on sphere

        u_mid            = FLOW.velocity(pos_mid, time + DT/2)
        omega_mid, _     = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        grad_u_mid       = FLOW.gradient(pos_mid, time + DT/2)

        n_mid    = E_Z @ expm(tau * grad_u_mid)
        hatn_mid = n_mid / jnp.linalg.norm(n_mid)

        dhatp_mid = (jnp.cross(omega_mid, hatp_mid)
                     + (hatn_mid - jnp.dot(hatn_mid, hatp_mid) * hatp_mid) / (2.0 * TAU_ALIGN))
        dpos_mid  = u_mid + hatp_mid

        position = position + DT * (dpos + dpos_mid) / 2
        hatp     = hatp     + DT * (dhatp + dhatp_mid) / 2
        hatp     = hatp     / jnp.linalg.norm(hatp)   # project back to S²

        return (position, hatp), position   # save position for trajectory

    (position, hatp), positions = jax.lax.scan(
        step, (position, hatp), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME, positions   # mean Z velocity + full path


def mean_velocity_surfer(tau):
    velocities  = jnp.stack([_rollout_surfer(tau, x)[0] for x in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)

grad_fn_surfer = jax.jit(jax.value_and_grad(mean_velocity_surfer))

In [17]:
initial_guess = 0.5 * jnp.ones(1)

optimal_surfer, optimal_velocity = gradient_descent(
    grad_fn_surfer,
    initial_guess,
    lr          = 0.01,
    n_steps     = 1000,
    print_every = 100,
    clip_min=0.01,
    clip_max=2.0,
)

step    0  velocity=1.20358  |grad|=0.02827  d0=0.5003
step  100  velocity=1.20391  |grad|=0.00980  d0=0.5177
step  200  velocity=1.20395  |grad|=0.00334  d0=0.5236
step  300  velocity=1.20396  |grad|=0.00113  d0=0.5257
step  400  velocity=1.20396  |grad|=0.00038  d0=0.5263
step  500  velocity=1.20396  |grad|=0.00013  d0=0.5266
step  600  velocity=1.20396  |grad|=0.00004  d0=0.5267
step  700  velocity=1.20396  |grad|=0.00001  d0=0.5267
step  800  velocity=1.20396  |grad|=0.00001  d0=0.5267
step  900  velocity=1.20396  |grad|=0.00000  d0=0.5267
step  999  velocity=1.20396  |grad|=0.00000  d0=0.5267

Optimal design:
  radius: 0.5266  (init: 0.5000)
Mean velocity: 1.20396


### Simulations for plots

In [46]:
DT = 0.1
N_DT = int(FINAL_TIME / DT)

tensors = mybody.compute_fast_mobility(dofs=jnp.zeros(mybody.Ndof), design=optimal_design)
FORCE_INTENSITY = 1.0/tensors.M_H[2,-1]
FORCE = constant_scalar(value=FORCE_INTENSITY)

In [52]:
surfer_trajectories = []

for yinit in Y_INIT_POSITIONS:
    v, p = _rollout_surfer(tau=optimal_surfer, x_init=yinit)
    print("New initial position:", yinit, ", velocity:", v)
    surfer_trajectories.append(p[:, np.newaxis, :])
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in surfer_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

New initial position: 0.2243994752564138 , velocity: 1.0893388
New initial position: 1.0899403083882957 , velocity: 1.2582045
New initial position: 1.9554811415201774 , velocity: 1.2601522
New initial position: 2.821021974652059 , velocity: 1.1130496
New initial position: 3.686562807783941 , velocity: 1.1961696
New initial position: 4.552103640915822 , velocity: 1.2660059
New initial position: 5.417644474047704 , velocity: 1.2447933
End Z positions: ['4.36π', '5.03π', '5.04π', '4.45π', '4.78π', '5.06π', '4.98π']
Mean velocity: 1.2


In [ ]:
mybody.set_design_defaults(new_design=optimal_design)

trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=mybody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

OLD default param values: ['radius', 'spring_k'] [ 0.143  0.405]
NEW default param values: ['radius', 'spring_k'] [ 0.143  0.405]
New initial position 0.2243994752564138
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 1.0899403083882957
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 1.9554811415201774
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 2.821021974652059
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 3.686562807783941
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 4.552103640915822
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 5.417644474047704
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
End Z positions: ['3.62π', '4.81π', '5.52π', '5.19π', '5.49π', '5.39π

In [29]:
myrigidbody.set_design_defaults(new_design=[optimal_design[0]])

rigid_trajectories = []

for x_init in Y_INIT_POSITIONS:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=myrigidbody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    rigid_trajectories.append(solver.simulate(final_time=FINAL_TIME))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in rigid_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/FINAL_TIME:.3g}")

OLD default param values: ['radius'] [ 0.143]
NEW default param values: ['radius'] [ 0.143]
New initial position 0.2243994752564138
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 1.0899403083882957
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 1.9554811415201774
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 2.821021974652059
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 3.686562807783941
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 4.552103640915822
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
New initial position 5.417644474047704
Time: 0.000 / 12.566  Integrator RK2
Time: 10.000 / 12.566  Integrator RK2
End Z positions: ['0.103π', '0.416π', '3.92π', '3.26π', '5.46π', '2.04π', '0.095π']
Mean velocity: 0.546


### Preparing the figure of trajectory

In [30]:
# --- Helper to build tick labels like "−2π", "−π/2", "0", "π", etc. ---
def pi_ticks(lo, hi, step_pi=1):
    """Return (tickvals, ticktext) with spacing of step_pi * π."""
    import math
    n_lo = math.floor(lo / (step_pi * np.pi))
    n_hi = math.ceil(hi  / (step_pi * np.pi))
    vals, texts = [], []
    for n in range(n_lo, n_hi + 1):
        v = n * step_pi * np.pi
        vals.append(v)
        k = n * step_pi          # multiple of π
        if   k == 0:  texts.append('0')
        elif k == 1:  texts.append('π')
        elif k == -1: texts.append('−π')
        else:         texts.append(f'{k}π')
    return vals, texts

def bounds_2pi(lo, hi, step_pi=1):
    """Round lo down and hi up to the nearest multiple of step_pi*π."""
    unit = step_pi * np.pi
    return np.floor(lo / unit) * unit, np.ceil(hi / unit) * unit

In [58]:
colors=[BLUEACCENT, REDACCENT, GREY]

# --- Compute global bounds across ALL trajectories ---
all_y = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 1]) for traj in trajectories + rigid_trajectories + surfer_trajectories])
all_z = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 2]) for traj in trajectories + rigid_trajectories + surfer_trajectories])

Y_MIN, Y_MAX = bounds_2pi(all_y.min(), all_y.max())
Z_MIN, Z_MAX = bounds_2pi(all_z.min(), all_z.max())

GRID_STEP = 1
y_ticks, _ = pi_ticks(Y_MIN, Y_MAX, step_pi=GRID_STEP)
z_ticks, _ = pi_ticks(Z_MIN, Z_MAX, step_pi=GRID_STEP)

ASPECT     = (Y_MAX - Y_MIN)
FIG_WIDTH  = 1000   # total width for both panels
FIG_HEIGHT = int((FIG_WIDTH / 3) * (Z_MAX - Z_MIN) / ASPECT)

fig = make_subplots(rows=1, cols=3, subplot_titles=("rigid", "soft", "surfer"),
                    horizontal_spacing=0.05)

#
ny, nz = 300, 300
y_grid = np.linspace(Y_MIN, Y_MAX, ny)
z_grid = np.linspace(Z_MIN, Z_MAX, nz)
Y, Z   = np.meshgrid(y_grid, z_grid)

omega = 1.0
PSI   = 0.5 * omega * np.sin(Y) * np.sin(Z)

# --- Streamlines: add to all subplots ---
for col in [1, 2, 3]:
    fig.add_trace(go.Contour(
        x=y_grid, y=z_grid, z=PSI,
        ncontours = 8,
        contours  = dict(coloring='none', showlines=True),
        line      = dict(color=GREY, width=0.8),
        showscale = False,
        showlegend= False,
        hoverinfo = 'skip',
    ), row=1, col=col)

# --- Helper to add one set of trajectories to a given col ---
def add_trajectories(fig, traj_list, col):
    for (trajectory, x_init) in zip(traj_list, Y_INIT_POSITIONS):
        positions = jnp.array([t[0] for t in trajectory])
        y_pos = np.array(positions[:, 1])
        z_pos = np.array(positions[:, 2])
        label = f"y₀={x_init/np.pi:.2g}π"

        fig.add_trace(go.Scatter(
            x=y_pos, y=z_pos,
            mode='lines',
            line=dict(color=colors[col-1], width=2),
            name=label, showlegend=False,
            hovertemplate=f"Y=%{{x:.3f}}<br>Z=%{{y:.3f}}<extra>{label}</extra>",
        ), row=1, col=col)
        
        fig.add_trace(go.Scatter(
            x=[y_pos[-1]], y=[z_pos[-1]],
            mode='markers',
            marker=dict(color=colors[col-1], size=8, symbol=['circle', 'square']),
            showlegend=False,
            hovertemplate=f"%{{text}}<extra>{label}</extra>",
            text=['start', 'end'],
        ), row=1, col=col)

add_trajectories(fig, rigid_trajectories,  col=1)
add_trajectories(fig, trajectories,        col=2)
add_trajectories(fig, surfer_trajectories, col=3)

# --- Shared axis style ---
axis_style = dict(
    showticklabels=False,
    showgrid=True, gridcolor=GREY, gridwidth=1,
    zeroline=True, linecolor=GREY, mirror=True,
    title=None,
    constrain='domain',
)

fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=20, r=20, t=30, b=20),
    showlegend=False,
    xaxis=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y',  scaleratio=1, tickvals=y_ticks),
    yaxis=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    xaxis2=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y2', scaleratio=1, tickvals=y_ticks),
    yaxis2=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    xaxis3=dict(**axis_style, range=[Y_MIN, Y_MAX], scaleanchor='y3', scaleratio=1, tickvals=y_ticks),
    yaxis3=dict(**axis_style, range=[Z_MIN, Z_MAX], constraintoward='center', tickvals=z_ticks),
    )


### Figure

In [ ]:
fig.show()

### Surfer

In [ ]:
from jax.scipy.linalg import expm

E_Z = jnp.array([0.0, 0.0, 1.0])   # reference "upward" direction

def _rollout_surfer(tau, talign, x_init):
    """Surfer agent: swims at speed 1, aligns with z @ exp(tau * grad_u)."""

    position = jnp.array([np.pi/2, x_init, 0.0])
    hatp     = E_Z.copy()   # initial swimming direction: upward

    def step(carry, t):
        position, hatp = carry
        time = t * DT

        u_lab              = FLOW.velocity(position, time)
        omega_lab, _       = FLOW.omega_rate_of_strain(position, time)
        grad_u             = FLOW.gradient(position, time)   # 3×3 matrix

        # --- Preferred direction ---
        n    = E_Z @ expm(tau * grad_u)         # deform current heading
        hatn = n / jnp.linalg.norm(n)           # normalize

        # --- hatp ODE (on the unit sphere) ---
        #   d hatp/dt = omega × hatp + (hatn - (hatn·hatp) hatp) / (2 talign)
        vorticity_term  = jnp.cross(omega_lab, hatp)
        alignment_term  = (hatn - jnp.dot(hatn, hatp) * hatp) / (2.0 * talign)
        dhatp           = vorticity_term + alignment_term

        # --- Position ODE: advected by flow + swimming at speed 1 ---
        dpos = u_lab + hatp

        # --- RK2 ---
        pos_mid  = position + DT * dpos  / 2
        hatp_mid = hatp     + DT * dhatp / 2
        hatp_mid = hatp_mid / jnp.linalg.norm(hatp_mid)   # keep on sphere

        u_mid            = FLOW.velocity(pos_mid, time + DT/2)
        omega_mid, _     = FLOW.omega_rate_of_strain(pos_mid, time + DT/2)
        grad_u_mid       = FLOW.gradient(pos_mid, time + DT/2)

        n_mid    = E_Z @ expm(tau * grad_u_mid)
        hatn_mid = n_mid / jnp.linalg.norm(n_mid)

        dhatp_mid = (jnp.cross(omega_mid, hatp_mid)
                     + (hatn_mid - jnp.dot(hatn_mid, hatp_mid) * hatp_mid) / (2.0 * talign))
        dpos_mid  = u_mid + hatp_mid

        position = position + DT * (dpos + dpos_mid) / 2
        hatp     = hatp     + DT * (dhatp + dhatp_mid) / 2
        hatp     = hatp     / jnp.linalg.norm(hatp)   # project back to S²

        return (position, hatp), position   # save position for trajectory

    (position, hatp), positions = jax.lax.scan(
        step, (position, hatp), jnp.arange(N_DT)
    )

    return position[2] / FINAL_TIME, positions   # mean Z velocity + full path


def mean_velocity_surfer(tau, talign):
    velocities  = jnp.stack([_rollout_surfer(tau, talign, x)[0] for x in Y_INIT_POSITIONS])
    return -jnp.mean(velocities)

grad_fn_surfer = jax.jit(jax.value_and_grad(mean_velocity_surfer))

0.26411396
1.0418978
1.2958483
0.8171326
0.8720813
1.3087085
0.7090508
